# Hybrid retrieval: BM25 plus dense plus reranker

The customer-support team's RAG bot retrieves with cosine similarity over `text-embedding-3-small`. Recall on the eval set is 62%. The lead asks: can we get to 85% without swapping the embedding model? You have 90 minutes to prove or disprove a hybrid retrieval pipeline combining BM25 (OpenSearch Serverless) and dense embeddings (Supabase pgvector), fused with reciprocal rank fusion, then reranked over the top-50 with Cohere Rerank 3. Measure recall@5 against a 100-query labeled eval set and target a 20-point lift.

Estimated time: 90 minutes (session window: 120 minutes). Cost: about $1.50 to $2.00 on a clean run; up to $3.00 if you walk away mid-lab and the OpenSearch collection burns a few extra hours.

OpenSearch Serverless costs about a dollar an hour. The cleanup cell deletes the collection so the bill stops. A 2-hour watchdog Lambda is your backup if you walk away. A clean session lands around two dollars.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. labrun-checks stays on 0.7.0 per the rest of the track.
# OpenSearch is queried with opensearch-py + requests-aws4auth (SigV4 for the
# Serverless aoss service). Cohere is the primary reranker.

!pip install --quiet labrun-checks==0.7.0 boto3==1.35.92 opensearch-py==2.7.1 requests-aws4auth==1.3.1 openai==1.59.7 supabase==2.10.0 psycopg2-binary==2.9.10 cohere==5.13.8 tenacity==9.0.0

In [ ]:
# Imports and per-lab constants. OpenSearch Serverless collection names cap
# at 32 chars, so we truncate the descriptor and rely on the labrun:lab-slug
# tag for orphan detection. Postgres identifiers use snake_case under the
# labrun_ prefix.

import atexit
import getpass
import json
import math
import os
import random
import re
import sys
import time
import uuid
import zipfile
from collections import defaultdict
from io import BytesIO

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "hybrid-search-bm25-dense-rerank"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

COLLECTION_NAME = "labrun-hybrid-rag-bm25"
ENCRYPTION_POLICY_NAME = "labrun-hybrid-rag-enc"
NETWORK_POLICY_NAME = "labrun-hybrid-rag-net"
ACCESS_POLICY_NAME = "labrun-hybrid-rag-access"
INDEX_NAME = "chunks"

PGVECTOR_TABLE = "labrun_hybrid_search_bm25_dense_rerank_chunks"

# 2-hour auto-destroy safety net (RESOURCE-SAFETY-SPEC.md Section 3 Lab 6
# pattern, adapted for an OpenSearch Serverless collection).
WATCHDOG_RULE_NAME = "labrun-hybrid-search-bm25-dense-rerank-cleanup-rule"
WATCHDOG_LAMBDA_NAME = "labrun-hybrid-search-bm25-dense-rerank-cleanup-fn"
WATCHDOG_ROLE_NAME = "labrun-hybrid-search-bm25-dense-rerank-cleanup-role"

EVAL_RESULTS_PATH = "/content/eval_results.json"

SEED = 1234

EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536

COHERE_RERANK_MODEL = "rerank-english-v3.0"

RRF_K = 60

In [ ]:
# NBVAL_SKIP
# BYOK setup. Session token, OpenAI key, AWS keys, Supabase URL +
# service-role key, Cohere key, optional Voyage key. Validate each
# before touching the cloud.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")

OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ").strip()
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://xxx.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
COHERE_API_KEY = getpass.getpass("COHERE_API_KEY: ").strip()
VOYAGE_API_KEY = getpass.getpass("VOYAGE_API_KEY (optional fallback, hit Enter to skip): ").strip()

required = {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY_ID,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "COHERE_API_KEY": COHERE_API_KEY,
}
missing = [k for k, v in required.items() if not v]
if missing:
    print(f"Missing required credentials: {missing}. Re-run this cell.")
    raise SystemExit(1)

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION

try:
    _sts = boto3.client(
        "sts",
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=REGION,
    )
    _ident = _sts.get_caller_identity()
    AWS_ACCOUNT_ID = _ident["Account"]
    print(f"AWS credentials validated. Account: {AWS_ACCOUNT_ID}, Region: {REGION}.")
except ClientError as exc:
    print(f"AWS credentials rejected: {exc!r}")
    print("Refresh AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY and re-run this cell.")
    raise SystemExit(1)

import cohere
try:
    _co_client = cohere.ClientV2(api_key=COHERE_API_KEY)
    _models = _co_client.models.list()
    print("Cohere credentials validated.")
except Exception as exc:
    print(f"Cohere key rejected: {exc!r}")
    print("Confirm COHERE_API_KEY has credit on dashboard.cohere.com and re-run this cell.")
    raise SystemExit(1)

from openai import OpenAI
try:
    _oai_client = OpenAI(api_key=OPENAI_API_KEY)
    _oai_client.models.list()
    print("OpenAI credentials validated.")
except Exception as exc:
    print(f"OpenAI key rejected: {exc!r}")
    print("Refresh OPENAI_API_KEY and re-run this cell.")
    raise SystemExit(1)

from supabase import create_client
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

DB_URI = getpass.getpass(
    "Supabase Postgres connection string "
    "(Project Settings > Database > Connection string > URI; "
    "use the Transaction pooler or Session pooler URL): "
).strip()

import psycopg2
try:
    _conn = psycopg2.connect(DB_URI)
    _conn.autocommit = True
    with _conn.cursor() as _cur:
        _cur.execute("SELECT 1")
    _conn.close()
    print("Supabase Postgres connection validated.")
except Exception as exc:
    print(f"Postgres connection failed: {exc!r}")
    print("Re-check the connection string (must include the password; port 6543 pooler recommended).")
    raise SystemExit(1)

CREDS = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region": REGION,
    "supabase_url": SUPABASE_URL,
    "service_role_key": SUPABASE_SERVICE_ROLE_KEY,
    "db_uri": DB_URI,
    "openai_api_key": OPENAI_API_KEY,
    "cohere_api_key": COHERE_API_KEY,
    "voyage_api_key": VOYAGE_API_KEY,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=CREDS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom adapter for OpenSearch Serverless + Supabase +
# Lambda + EventBridge + IAM + local files, atexit safety net, orphan scan.
#
# Manifest order is critical-first per RESOURCE-SAFETY-SPEC.md Section 2
# Rule 2: the OpenSearch Serverless collection tears down before anything
# else because OCU billing is the only hourly surface in this lab. The
# three security policies follow because deleting a policy while the
# collection still references it is a 400.
#
# TODO: extend labrun-checks aws.py with aoss helpers and vector_db.py
# for pgvector so future labs can reuse this adapter without copying it.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="aoss_collection",
        id=COLLECTION_NAME,
        region=REGION,
        critical=True,
        cli_delete_command=(
            f"aws opensearchserverless delete-collection --id "
            f"$(aws opensearchserverless batch-get-collection --names {COLLECTION_NAME} "
            f"--query 'collectionDetails[0].id' --output text) --region {REGION}"
        ),
    ),
    CleanupResource(
        type="aoss_security_policy",
        id=ACCESS_POLICY_NAME,
        region=REGION,
        extra={"policy_type": "data"},
        cli_delete_command=(
            f"aws opensearchserverless delete-access-policy --name {ACCESS_POLICY_NAME} "
            f"--type data --region {REGION}"
        ),
    ),
    CleanupResource(
        type="aoss_security_policy",
        id=NETWORK_POLICY_NAME,
        region=REGION,
        extra={"policy_type": "network"},
        cli_delete_command=(
            f"aws opensearchserverless delete-security-policy --name {NETWORK_POLICY_NAME} "
            f"--type network --region {REGION}"
        ),
    ),
    CleanupResource(
        type="aoss_security_policy",
        id=ENCRYPTION_POLICY_NAME,
        region=REGION,
        extra={"policy_type": "encryption"},
        cli_delete_command=(
            f"aws opensearchserverless delete-security-policy --name {ENCRYPTION_POLICY_NAME} "
            f"--type encryption --region {REGION}"
        ),
    ),
    CleanupResource(
        type="events_rule",
        id=WATCHDOG_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {WATCHDOG_RULE_NAME} --ids 1 --region {REGION} && "
            f"aws events delete-rule --name {WATCHDOG_RULE_NAME} --region {REGION}"
        ),
    ),
    CleanupResource(
        type="lambda_function",
        id=WATCHDOG_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=f"aws lambda delete-function --function-name {WATCHDOG_LAMBDA_NAME} --region {REGION}",
    ),
    CleanupResource(
        type="iam_role",
        id=WATCHDOG_ROLE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws iam detach-role-policy --role-name {WATCHDOG_ROLE_NAME} "
            f"--policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole; "
            f"aws iam delete-role-policy --role-name {WATCHDOG_ROLE_NAME} --policy-name inline; "
            f"aws iam delete-role --role-name {WATCHDOG_ROLE_NAME}"
        ),
    ),
    CleanupResource(
        type="supabase_table",
        id=PGVECTOR_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{PGVECTOR_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="local_file",
        id=EVAL_RESULTS_PATH,
        region="local",
        cli_delete_command=f"rm -f {EVAL_RESULTS_PATH}",
    ),
]


class HybridLabCleanupAdapter:
    """Custom adapter spanning OpenSearch Serverless, Supabase, Lambda,
    EventBridge, IAM, and local files. Each branch swallows already-gone
    errors so cleanup is idempotent.

    TODO: split into labrun-checks aws.py (aoss + lambda + events + iam)
    and vector_db.py (pgvector) once the API stabilises across labs.
    """

    def __init__(self, credentials):
        self._creds = credentials
        self._db_uri = credentials["db_uri"]

    def _aoss(self):
        return boto3.client("opensearchserverless",
            aws_access_key_id=self._creds["aws_access_key_id"],
            aws_secret_access_key=self._creds["aws_secret_access_key"],
            region_name=self._creds["region"])

    def _events(self):
        return boto3.client("events",
            aws_access_key_id=self._creds["aws_access_key_id"],
            aws_secret_access_key=self._creds["aws_secret_access_key"],
            region_name=self._creds["region"])

    def _lambda(self):
        return boto3.client("lambda",
            aws_access_key_id=self._creds["aws_access_key_id"],
            aws_secret_access_key=self._creds["aws_secret_access_key"],
            region_name=self._creds["region"])

    def _iam(self):
        return boto3.client("iam",
            aws_access_key_id=self._creds["aws_access_key_id"],
            aws_secret_access_key=self._creds["aws_secret_access_key"])

    def _tagging(self):
        return boto3.client("resourcegroupstaggingapi",
            aws_access_key_id=self._creds["aws_access_key_id"],
            aws_secret_access_key=self._creds["aws_secret_access_key"],
            region_name=self._creds["region"])

    def _exec_sql(self, sql):
        conn = psycopg2.connect(self._db_uri)
        conn.autocommit = True
        try:
            with conn.cursor() as cur:
                cur.execute(sql)
        finally:
            conn.close()

    def _table_exists(self, table_name):
        conn = psycopg2.connect(self._db_uri)
        try:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT 1 FROM information_schema.tables "
                    "WHERE table_schema = 'public' AND table_name = %s",
                    (table_name,),
                )
                return cur.fetchone() is not None
        finally:
            conn.close()

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "aoss_collection":
            aoss = self._aoss()
            try:
                resp = aoss.batch_get_collection(names=[rid])
                details = resp.get("collectionDetails", [])
                if not details:
                    return
                coll_id = details[0]["id"]
                aoss.delete_collection(id=coll_id)
                deadline = time.time() + 300
                while time.time() < deadline:
                    resp2 = aoss.batch_get_collection(names=[rid])
                    if not resp2.get("collectionDetails"):
                        return
                    status = resp2["collectionDetails"][0].get("status")
                    if status == "DELETED":
                        return
                    time.sleep(15)
            except ClientError as exc:
                code_ = exc.response.get("Error", {}).get("Code", "")
                if code_ in ("ResourceNotFoundException",):
                    return
                raise
        elif rtype == "aoss_security_policy":
            aoss = self._aoss()
            ptype = (resource.extra or {}).get("policy_type", "encryption")
            try:
                if ptype == "data":
                    aoss.delete_access_policy(name=rid, type="data")
                else:
                    aoss.delete_security_policy(name=rid, type=ptype)
            except ClientError as exc:
                code_ = exc.response.get("Error", {}).get("Code", "")
                if code_ in ("ResourceNotFoundException",):
                    return
                raise
        elif rtype == "events_rule":
            ev = self._events()
            try:
                targets = ev.list_targets_by_rule(Rule=rid).get("Targets", [])
                if targets:
                    ev.remove_targets(Rule=rid, Ids=[t["Id"] for t in targets])
                ev.delete_rule(Name=rid)
            except ClientError as exc:
                code_ = exc.response.get("Error", {}).get("Code", "")
                if code_ in ("ResourceNotFoundException",):
                    return
                raise
        elif rtype == "lambda_function":
            la = self._lambda()
            try:
                la.delete_function(FunctionName=rid)
            except ClientError as exc:
                code_ = exc.response.get("Error", {}).get("Code", "")
                if code_ in ("ResourceNotFoundException",):
                    return
                raise
        elif rtype == "iam_role":
            iam = self._iam()
            try:
                attached = iam.list_attached_role_policies(RoleName=rid).get("AttachedPolicies", [])
                for ap in attached:
                    iam.detach_role_policy(RoleName=rid, PolicyArn=ap["PolicyArn"])
                inline = iam.list_role_policies(RoleName=rid).get("PolicyNames", [])
                for pn in inline:
                    iam.delete_role_policy(RoleName=rid, PolicyName=pn)
                iam.delete_role(RoleName=rid)
            except ClientError as exc:
                code_ = exc.response.get("Error", {}).get("Code", "")
                if code_ in ("NoSuchEntity", "NoSuchEntityException"):
                    return
                raise
        elif rtype == "supabase_table":
            self._exec_sql(f"DROP TABLE IF EXISTS public.{rid} CASCADE")
        elif rtype == "local_file":
            try:
                os.remove(rid)
            except FileNotFoundError:
                pass
        else:
            raise ValueError(f"HybridLabCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        try:
            if rtype == "aoss_collection":
                resp = self._aoss().batch_get_collection(names=[rid])
                details = resp.get("collectionDetails", [])
                return bool(details) and details[0].get("status") != "DELETED"
            if rtype == "aoss_security_policy":
                aoss = self._aoss()
                ptype = (resource.extra or {}).get("policy_type", "encryption")
                try:
                    if ptype == "data":
                        aoss.get_access_policy(name=rid, type="data")
                    else:
                        aoss.get_security_policy(name=rid, type=ptype)
                    return True
                except ClientError as exc:
                    if exc.response.get("Error", {}).get("Code") == "ResourceNotFoundException":
                        return False
                    raise
            if rtype == "events_rule":
                try:
                    self._events().describe_rule(Name=rid)
                    return True
                except ClientError as exc:
                    if exc.response.get("Error", {}).get("Code") == "ResourceNotFoundException":
                        return False
                    raise
            if rtype == "lambda_function":
                try:
                    self._lambda().get_function(FunctionName=rid)
                    return True
                except ClientError as exc:
                    if exc.response.get("Error", {}).get("Code") == "ResourceNotFoundException":
                        return False
                    raise
            if rtype == "iam_role":
                try:
                    self._iam().get_role(RoleName=rid)
                    return True
                except ClientError as exc:
                    code_ = exc.response.get("Error", {}).get("Code")
                    if code_ in ("NoSuchEntity", "NoSuchEntityException"):
                        return False
                    raise
            if rtype == "supabase_table":
                return self._table_exists(rid)
            if rtype == "local_file":
                return os.path.exists(rid)
        except Exception:
            return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        if region == self._creds["region"]:
            try:
                paginator = self._tagging().get_paginator("get_resources")
                for page in paginator.paginate(
                    TagFilters=[{"Key": "labrun:lab-slug", "Values": [lab_slug]}],
                ):
                    for r in page.get("ResourceTagMappingList", []):
                        arns.append(r["ResourceARN"])
            except Exception:
                pass
        if region == "supabase":
            try:
                conn = psycopg2.connect(self._db_uri)
                try:
                    with conn.cursor() as cur:
                        cur.execute(
                            "SELECT table_name FROM information_schema.tables "
                            "WHERE table_schema = 'public' AND table_name LIKE %s",
                            ("labrun_hybrid_search_bm25_dense_rerank_%",),
                        )
                        for row in cur.fetchall():
                            arns.append(row[0])
                finally:
                    conn.close()
            except Exception:
                pass
        return arns


CLEANUP_ADAPTER = HybridLabCleanupAdapter(credentials=CREDS)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    aws_orphans = CLEANUP_ADAPTER.tag_scan(CREDS, LAB_TAG_VALUE, REGION)
    supa_orphans = CLEANUP_ADAPTER.tag_scan(CREDS, LAB_TAG_VALUE, "supabase")
    return aws_orphans, supa_orphans


_aws_orphans, _supa_orphans = scan_for_orphans()
if _aws_orphans or _supa_orphans:
    print("BLOCKED: orphan resources from a previous session were found.")
    for arn in _aws_orphans:
        print("  AWS:", arn)
    for t in _supa_orphans:
        print("  Supabase table:", t)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision resources for this session.")

## Task 1: Stand up the OpenSearch Serverless collection and index the corpus

This task creates the BM25 half of the hybrid pipeline. The three security policies (encryption, network, data access) all attach to the collection by name; missing any one of them produces a 403 from the data plane even after the collection reports ACTIVE.

The 2-hour watchdog (EventBridge rule + Lambda + IAM role) is created BEFORE the collection so the watchdog is armed before any OCU clock starts ticking. The Lambda payload includes the collection name and the three policy names so the watchdog can clean up on its own if the cleanup cell never runs.

You will also seed a small synthetic markdown corpus (~2 MB), chunk it into roughly 300 chunks with `chunk_<index>` IDs, and bulk-index the chunks with `helpers.bulk()`.

In [ ]:
# Task 1: provision the watchdog, create the OpenSearch Serverless collection,
# wait for ACTIVE, build the synthetic corpus, and bulk-index the chunks.

from opensearchpy import OpenSearch, RequestsHttpConnection, helpers
from requests_aws4auth import AWS4Auth

# ---- Synthetic corpus generator (deterministic seed) ----------------------
random.seed(SEED)

TOPICS = [
    ("billing", "How do I update the credit card on file?"),
    ("auth", "I am getting a 401 on the login endpoint."),
    ("export", "How do I export my data as a CSV?"),
    ("teams", "How do I invite a teammate to my workspace?"),
    ("integrations", "How do I connect Slack to my workspace?"),
    ("api", "What is the rate limit on the API?"),
    ("storage", "How much storage does my plan include?"),
    ("notifications", "How do I mute notifications for one project?"),
    ("admin", "How do I change a project owner?"),
    ("plans", "What is included in the pro plan?"),
]

def _build_corpus():
    chunks = []
    cid = 0
    for topic, _ in TOPICS:
        for i in range(30):
            cid += 1
            body = (
                f"# {topic} reference note {i}\n\n"
                f"This note covers {topic} configuration step {i}. "
                f"Field name: {topic}_{i}_field. "
                f"Procedure: open settings, locate {topic} section, click edit, save. "
                f"Common pitfalls: invalid {topic} input, expired session token, "
                f"rate-limit at 60 requests/minute, missing scope on the API key. "
                f"Related docs: see {topic}_overview, {topic}_recipes, {topic}_troubleshooting. "
                f"Tag: {topic}-{i:02d}."
            )
            chunks.append({"chunk_id": f"chunk_{cid}", "text": body, "topic": topic})
    return chunks

CORPUS = _build_corpus()
print(f"Corpus built: {len(CORPUS)} chunks, roughly {sum(len(c['text']) for c in CORPUS) // 1024} KB.")

# ---- Watchdog (IAM + Lambda + EventBridge) --------------------------------
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

# YOUR CODE: create the IAM role WATCHDOG_ROLE_NAME with trust_policy above,
# YOUR CODE:   tagged with labrun:lab-slug = LAB_TAG_VALUE. Assign the role
# YOUR CODE:   ARN to watchdog_role_arn.
# YOUR CODE: attach AWSLambdaBasicExecutionRole (managed) so the Lambda can log.
# YOUR CODE: put an inline policy granting aoss:DeleteCollection,
# YOUR CODE:   aoss:DeleteAccessPolicy, aoss:DeleteSecurityPolicy,
# YOUR CODE:   aoss:BatchGetCollection on resource '*'.

watchdog_role_arn = None

print("Waiting 20s for IAM role propagation before creating the Lambda...")
time.sleep(20)

LAMBDA_CODE = """
import json, time, boto3
def handler(event, context):
    aoss = boto3.client('opensearchserverless')
    name = event['collection_name']
    resp = aoss.batch_get_collection(names=[name])
    details = resp.get('collectionDetails', [])
    if details:
        cid = details[0]['id']
        aoss.delete_collection(id=cid)
        deadline = time.time() + 300
        while time.time() < deadline:
            r2 = aoss.batch_get_collection(names=[name])
            if not r2.get('collectionDetails'):
                break
            if r2['collectionDetails'][0].get('status') == 'DELETED':
                break
            time.sleep(15)
    for pname, ptype in [
        (event['access_policy'], 'data'),
        (event['network_policy'], 'network'),
        (event['encryption_policy'], 'encryption'),
    ]:
        try:
            if ptype == 'data':
                aoss.delete_access_policy(name=pname, type='data')
            else:
                aoss.delete_security_policy(name=pname, type=ptype)
        except Exception:
            pass
    return {'status': 'watchdog-fired', 'collection': name}
"""

zip_buf = BytesIO()
with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", LAMBDA_CODE)
zip_bytes = zip_buf.getvalue()

la = boto3.client(
    "lambda",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION,
)

# YOUR CODE: la.create_function(FunctionName=WATCHDOG_LAMBDA_NAME,
# YOUR CODE:   Runtime='python3.11', Role=watchdog_role_arn,
# YOUR CODE:   Handler='lambda_function.handler',
# YOUR CODE:   Code={'ZipFile': zip_bytes}, Timeout=600,
# YOUR CODE:   Tags={'labrun:lab-slug': LAB_TAG_VALUE})

ev = boto3.client(
    "events",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION,
)

# EventBridge schedule: fire once at NOW + 2 hours.
fire_at = time.gmtime(time.time() + 2 * 3600)
schedule_expr = time.strftime("cron(%M %H %d %m ? %Y)", fire_at)

# YOUR CODE: ev.put_rule(Name=WATCHDOG_RULE_NAME, ScheduleExpression=schedule_expr,
# YOUR CODE:   State='ENABLED',
# YOUR CODE:   Tags=[{'Key': 'labrun:lab-slug', 'Value': LAB_TAG_VALUE}])
# YOUR CODE: la.add_permission(FunctionName=WATCHDOG_LAMBDA_NAME, StatementId='evt',
# YOUR CODE:   Action='lambda:InvokeFunction', Principal='events.amazonaws.com',
# YOUR CODE:   SourceArn=f'arn:aws:events:{REGION}:{AWS_ACCOUNT_ID}:rule/{WATCHDOG_RULE_NAME}')
# YOUR CODE: ev.put_targets(Rule=WATCHDOG_RULE_NAME, Targets=[{
# YOUR CODE:   'Id': '1',
# YOUR CODE:   'Arn': f'arn:aws:lambda:{REGION}:{AWS_ACCOUNT_ID}:function:{WATCHDOG_LAMBDA_NAME}',
# YOUR CODE:   'Input': json.dumps({
# YOUR CODE:     'collection_name': COLLECTION_NAME,
# YOUR CODE:     'access_policy': ACCESS_POLICY_NAME,
# YOUR CODE:     'network_policy': NETWORK_POLICY_NAME,
# YOUR CODE:     'encryption_policy': ENCRYPTION_POLICY_NAME,
# YOUR CODE:   })}])

print(f"Watchdog armed: rule {WATCHDOG_RULE_NAME} will fire at {time.strftime('%Y-%m-%d %H:%M UTC', fire_at)}.")

# ---- OpenSearch Serverless collection -------------------------------------
aoss = boto3.client(
    "opensearchserverless",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION,
)

encryption_policy = {
    "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}],
    "AWSOwnedKey": True,
}
network_policy = [{
    "Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
        {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]},
    ],
    "AllowFromPublic": True,
}]
access_policy = [{
    "Rules": [
        {"ResourceType": "index", "Resource": [f"index/{COLLECTION_NAME}/*"],
         "Permission": ["aoss:*"]},
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"],
         "Permission": ["aoss:*"]},
    ],
    "Principal": [f"arn:aws:iam::{AWS_ACCOUNT_ID}:root",
                  f"arn:aws:iam::{AWS_ACCOUNT_ID}:user/labrun-test"],
}]

# YOUR CODE: aoss.create_security_policy(name=ENCRYPTION_POLICY_NAME,
# YOUR CODE:   type='encryption', policy=json.dumps(encryption_policy))
# YOUR CODE: aoss.create_security_policy(name=NETWORK_POLICY_NAME,
# YOUR CODE:   type='network', policy=json.dumps(network_policy))
# YOUR CODE: aoss.create_access_policy(name=ACCESS_POLICY_NAME,
# YOUR CODE:   type='data', policy=json.dumps(access_policy))
# YOUR CODE: aoss.create_collection(name=COLLECTION_NAME, type='SEARCH',
# YOUR CODE:   tags=[{'key': 'labrun:lab-slug', 'value': LAB_TAG_VALUE}])

print("Polling for collection ACTIVE state (5-10 minutes typical, your coffee will be cold)...")
deadline = time.time() + 900
endpoint = None
while time.time() < deadline:
    try:
        resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
        details = resp.get("collectionDetails", [])
        if details:
            status = details[0].get("status")
            if status == "ACTIVE":
                endpoint = details[0]["collectionEndpoint"]
                print(f"Collection ACTIVE at {endpoint}")
                break
            print(f"  status: {status}, waiting 30s...")
    except ClientError as exc:
        print(f"  poll error (continuing): {exc}")
    time.sleep(30)

if endpoint is None:
    print("Collection did not reach ACTIVE within 15 minutes. Re-run this cell or check the console.")
    raise SystemExit(1)

# ---- Bulk-index the corpus ------------------------------------------------
host = endpoint.replace("https://", "").rstrip("/")
awsauth = AWS4Auth(AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, REGION, "aoss")
os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=60,
)

mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "text": {"type": "text"},
            "topic": {"type": "keyword"},
        }
    }
}

# YOUR CODE: os_client.indices.create(index=INDEX_NAME, body=mapping)
# YOUR CODE: actions = [{'_index': INDEX_NAME, '_id': c['chunk_id'], '_source': c}
# YOUR CODE:            for c in CORPUS]
# YOUR CODE: success, failed = helpers.bulk(os_client, actions, raise_on_error=False)
# YOUR CODE: print(f'Indexed {success} docs, {len(failed)} failures')

os_client.indices.refresh(index=INDEX_NAME)
print(f"BM25 index {INDEX_NAME!r} is ready.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: collection ACTIVE AND chunks index doc count >= 200.

def checkpoint_1(session):
    try:
        aoss = boto3.client(
            "opensearchserverless",
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
            region_name=REGION,
        )
        resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
        details = resp.get("collectionDetails", [])
        if not details:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Collection {COLLECTION_NAME!r} not found. Did the create_collection "
                    f"call succeed? Check the AWS console under OpenSearch > Serverless."
                ),
            )
        status = details[0].get("status")
        if status != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Collection status is {status!r}, expected ACTIVE. OpenSearch Serverless "
                    f"provisioning takes 5-10 minutes; wait and re-run this checkpoint."
                ),
            )
        endpoint = details[0]["collectionEndpoint"]
        host = endpoint.replace("https://", "").rstrip("/")
        awsauth = AWS4Auth(AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, REGION, "aoss")
        client = OpenSearch(
            hosts=[{"host": host, "port": 443}],
            http_auth=awsauth, use_ssl=True, verify_certs=True,
            connection_class=RequestsHttpConnection, timeout=30,
        )
        try:
            client.indices.refresh(index=INDEX_NAME)
            count_resp = client.count(index=INDEX_NAME)
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not read /chunks/_count: {exc!r}. Either the access policy is not "
                    f"yet attached (403) or the index was never created."
                ),
            )
        n = count_resp.get("count", 0)
        if n < 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Chunks index has {n} docs; expected at least 200. Check helpers.bulk() "
                    f"for failed items, or wait for indexing to complete."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=repr(exc))

check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

OpenSearch Serverless collection or the chunks index is not where you expect. Check the AWS console: is the collection `ACTIVE`? Is the `chunks` index present? Are all three security policies (encryption, network, data access) attached?

</details>

<details><summary>Hint 2 (direction)</summary>

OpenSearch Serverless collection provisioning takes 5 to 10 minutes from `create_collection`. The waiter pattern is `aoss.batch_get_collection(names=[name])` polled every 30 seconds; do not start indexing before the collection is `ACTIVE` because the data plane will 403 (the access policy is attached lazily). The three policies must all exist before `create_collection`: encryption, network, then data access in that order.

For the watchdog: the IAM role needs 15-20 seconds to propagate before Lambda will accept it as a function role. The Lambda function tags must include `labrun:lab-slug` so the orphan scan picks it up if cleanup fails.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
role = iam.create_role(
    RoleName=WATCHDOG_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Tags=[{"Key": "labrun:lab-slug", "Value": LAB_TAG_VALUE}],
)
watchdog_role_arn = role["Role"]["Arn"]
iam.attach_role_policy(
    RoleName=WATCHDOG_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
iam.put_role_policy(
    RoleName=WATCHDOG_ROLE_NAME,
    PolicyName="inline",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["aoss:DeleteCollection", "aoss:DeleteAccessPolicy",
                       "aoss:DeleteSecurityPolicy", "aoss:BatchGetCollection"],
            "Resource": "*",
        }],
    }),
)

la.create_function(
    FunctionName=WATCHDOG_LAMBDA_NAME,
    Runtime="python3.11",
    Role=watchdog_role_arn,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=600,
    Tags={"labrun:lab-slug": LAB_TAG_VALUE},
)

ev.put_rule(
    Name=WATCHDOG_RULE_NAME,
    ScheduleExpression=schedule_expr,
    State="ENABLED",
    Tags=[{"Key": "labrun:lab-slug", "Value": LAB_TAG_VALUE}],
)
la.add_permission(
    FunctionName=WATCHDOG_LAMBDA_NAME,
    StatementId="evt",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=f"arn:aws:events:{REGION}:{AWS_ACCOUNT_ID}:rule/{WATCHDOG_RULE_NAME}",
)
ev.put_targets(
    Rule=WATCHDOG_RULE_NAME,
    Targets=[{
        "Id": "1",
        "Arn": f"arn:aws:lambda:{REGION}:{AWS_ACCOUNT_ID}:function:{WATCHDOG_LAMBDA_NAME}",
        "Input": json.dumps({
            "collection_name": COLLECTION_NAME,
            "access_policy": ACCESS_POLICY_NAME,
            "network_policy": NETWORK_POLICY_NAME,
            "encryption_policy": ENCRYPTION_POLICY_NAME,
        }),
    }],
)

aoss.create_security_policy(name=ENCRYPTION_POLICY_NAME, type="encryption",
    policy=json.dumps(encryption_policy))
aoss.create_security_policy(name=NETWORK_POLICY_NAME, type="network",
    policy=json.dumps(network_policy))
aoss.create_access_policy(name=ACCESS_POLICY_NAME, type="data",
    policy=json.dumps(access_policy))
aoss.create_collection(name=COLLECTION_NAME, type="SEARCH",
    tags=[{"key": "labrun:lab-slug", "value": LAB_TAG_VALUE}])

os_client.indices.create(index=INDEX_NAME, body=mapping)
actions = [{"_index": INDEX_NAME, "_id": c["chunk_id"], "_source": c} for c in CORPUS]
success, failed = helpers.bulk(os_client, actions, raise_on_error=False)
print(f"Indexed {success} docs, {len(failed)} failures")
```

</details>

**Wallet check.** OpenSearch Serverless collection is ACTIVE; OCU clock started at $0.96/hour (4 OCU minimum). Watchdog Lambda + EventBridge rule armed for the 2-hour mark. Spent so far: about 16 cents on OpenSearch. Bulk indexing was free against the same OCU pool. Your morning coffee cost 25x more.

## Task 2: Build the dense index in Supabase pgvector

The dense half of the pipeline lives in Supabase. You enable the `vector` extension, create the `labrun_hybrid_search_bm25_dense_rerank_chunks` table with a 1536-dimension column matching `text-embedding-3-small`, embed the corpus in batches of 50 (smaller batches stay under the OpenAI batch token limit and reduce 429 risk), and upsert by `chunk_id` so retries are idempotent.

The checkpoint compares row counts across OpenSearch and Supabase. A drift of more than 1 means the embedding loop crashed midway; rerun this cell, the upsert is idempotent.

In [ ]:
# NBVAL_SKIP
# Task 2: pgvector table + dense embedding loop with tenacity retries.

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# ---- Create the table -----------------------------------------------------
_conn = psycopg2.connect(DB_URI)
_conn.autocommit = True
with _conn.cursor() as _cur:
    _cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    _cur.execute(
        f"""
        CREATE TABLE IF NOT EXISTS public.{PGVECTOR_TABLE} (
            chunk_id text PRIMARY KEY,
            text text NOT NULL,
            topic text,
            embedding vector({EMBED_DIM})
        )
        """
    )
_conn.close()
print(f"pgvector table {PGVECTOR_TABLE} ready (dim={EMBED_DIM}).")

# ---- Embedding loop -------------------------------------------------------
oai = OpenAI(api_key=OPENAI_API_KEY)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type(Exception),
)
def _embed_batch(texts):
    resp = oai.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

BATCH = 50

# YOUR CODE: open a psycopg2 connection (autocommit=True) to DB_URI
# YOUR CODE: loop over CORPUS in slices of BATCH
# YOUR CODE:   call _embed_batch on the list of chunk text strings
# YOUR CODE:   for each (chunk, embedding) pair, run
# YOUR CODE:     INSERT INTO public.{PGVECTOR_TABLE} (chunk_id, text, topic, embedding)
# YOUR CODE:     VALUES (%s, %s, %s, %s::vector)
# YOUR CODE:     ON CONFLICT (chunk_id) DO UPDATE SET ...
# YOUR CODE: print a one-line progress message every batch

# ---- Verify the count -----------------------------------------------------
_conn = psycopg2.connect(DB_URI)
with _conn.cursor() as _cur:
    _cur.execute(f"SELECT count(*) FROM public.{PGVECTOR_TABLE}")
    pg_count = _cur.fetchone()[0]
_conn.close()
print(f"pgvector rows: {pg_count}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: pgvector row count matches OpenSearch _count within 1.

def checkpoint_2(session):
    try:
        aoss = boto3.client(
            "opensearchserverless",
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
            region_name=REGION,
        )
        details = aoss.batch_get_collection(names=[COLLECTION_NAME]).get("collectionDetails", [])
        if not details or details[0].get("status") != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason="OpenSearch collection is not ACTIVE; finish Checkpoint 1 first.",
            )
        endpoint = details[0]["collectionEndpoint"]
        host = endpoint.replace("https://", "").rstrip("/")
        awsauth = AWS4Auth(AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, REGION, "aoss")
        client = OpenSearch(
            hosts=[{"host": host, "port": 443}],
            http_auth=awsauth, use_ssl=True, verify_certs=True,
            connection_class=RequestsHttpConnection, timeout=30,
        )
        client.indices.refresh(index=INDEX_NAME)
        os_count = client.count(index=INDEX_NAME).get("count", 0)

        conn = psycopg2.connect(DB_URI)
        with conn.cursor() as cur:
            cur.execute(f"SELECT count(*) FROM public.{PGVECTOR_TABLE}")
            pg_count = cur.fetchone()[0]
        conn.close()

        diff = abs(os_count - pg_count)
        if diff > 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"OpenSearch has {os_count} docs but pgvector has {pg_count} (diff={diff}). "
                    f"Either the dense embedding loop crashed midway (re-run the embedding cell; "
                    f"the upsert is idempotent on chunk_id) or pgvector vector extension was not enabled."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=repr(exc))

check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

OpenSearch and pgvector have different row counts. One of the two indexers crashed midway. Print both counts side by side and compare. The most common cause is a 429 from the OpenAI embeddings endpoint when the batch is too large.

</details>

<details><summary>Hint 2 (direction)</summary>

If the dense embedding loop crashed, the most common cause is hitting an OpenAI 429 rate limit on the batch size. Lower your batch size to 50 chunks and rely on the `tenacity` `@retry` decorator already wired into `_embed_batch`. The OpenAI SDK does NOT retry 429s on embedding endpoints automatically.

For the upsert: `INSERT ... ON CONFLICT (chunk_id) DO UPDATE` makes the loop idempotent. The vector column accepts a list of floats via psycopg2 if you cast: `cur.execute("INSERT ... VALUES (..., %s::vector)", (chunk_id, text, topic, str(embedding)))`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
conn = psycopg2.connect(DB_URI)
conn.autocommit = True
total = 0
for i in range(0, len(CORPUS), BATCH):
    batch = CORPUS[i:i + BATCH]
    embeddings = _embed_batch([c["text"] for c in batch])
    with conn.cursor() as cur:
        for c, emb in zip(batch, embeddings):
            cur.execute(
                f"""
                INSERT INTO public.{PGVECTOR_TABLE} (chunk_id, text, topic, embedding)
                VALUES (%s, %s, %s, %s::vector)
                ON CONFLICT (chunk_id) DO UPDATE
                SET text = EXCLUDED.text,
                    topic = EXCLUDED.topic,
                    embedding = EXCLUDED.embedding
                """,
                (c["chunk_id"], c["text"], c["topic"], str(emb)),
            )
    total += len(batch)
    print(f"  embedded + upserted {total}/{len(CORPUS)}")
conn.close()
```

</details>

**Wallet check.** Embedded ~300 chunks at 1536 dims with `text-embedding-3-small`. Roughly 30K tokens billed at $0.02/1M = $0.0006. OpenAI cost so far: less than a penny. OpenSearch is at about 30 cents now (OCU clock has been running for ~20 minutes). Cohere has not been touched yet. Running total: roughly $0.30. Your coffee cost 13x more.

## Task 3: Reciprocal rank fusion over BM25 and dense top-50

Reciprocal rank fusion is one line of math: for each document, score = sum over each list of `1 / (k + rank_in_list)`. Three things to get right:

1. `k` is a smoothing constant; the published default is 60. Lower k (10-20) gives more weight to top results; higher k (>100) flattens.
2. `rank_in_list` is 1-indexed: the first result has rank 1, NOT rank 0.
3. If a document does not appear in a list, its contribution from that list is 0, NOT `1/k`.

You will write `hybrid_retrieve(query, k=50)` that returns 50 fused results sorted DESC by `rrf_score`. The checkpoint recomputes RRF independently and compares.

In [ ]:
# NBVAL_SKIP
# Task 3: hybrid_retrieve combining BM25 (OpenSearch) + dense (pgvector) via RRF.

_details = aoss.batch_get_collection(names=[COLLECTION_NAME])["collectionDetails"]
_endpoint = _details[0]["collectionEndpoint"]
_host = _endpoint.replace("https://", "").rstrip("/")
_awsauth = AWS4Auth(AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, REGION, "aoss")
os_search = OpenSearch(
    hosts=[{"host": _host, "port": 443}],
    http_auth=_awsauth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, timeout=30,
)


def bm25_top_n(query, n):
    resp = os_search.search(
        index=INDEX_NAME,
        body={"size": n, "query": {"match": {"text": query}}},
    )
    return [
        {"chunk_id": h["_source"]["chunk_id"],
         "chunk_text": h["_source"]["text"],
         "topic": h["_source"].get("topic"),
         "bm25_score": h["_score"]}
        for h in resp["hits"]["hits"]
    ]


def dense_top_n(query, n):
    emb = oai.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    conn = psycopg2.connect(DB_URI)
    try:
        with conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT chunk_id, text, topic, 1 - (embedding <=> %s::vector) AS sim
                FROM public.{PGVECTOR_TABLE}
                ORDER BY embedding <=> %s::vector
                LIMIT %s
                """,
                (str(emb), str(emb), n),
            )
            rows = cur.fetchall()
    finally:
        conn.close()
    return [{"chunk_id": r[0], "chunk_text": r[1], "topic": r[2], "dense_score": float(r[3])}
            for r in rows]


def hybrid_retrieve(query, k=50):
    # YOUR CODE: bm25 = bm25_top_n(query, k)
    # YOUR CODE: dense = dense_top_n(query, k)
    # YOUR CODE: build {chunk_id: bm25_rank} (1-indexed) and {chunk_id: dense_rank} (1-indexed)
    # YOUR CODE: build a union of chunk_ids across both lists
    # YOUR CODE: for each chunk_id, compute rrf_score
    # YOUR CODE:   = (1/(RRF_K + bm25_rank) if present else 0)
    # YOUR CODE:   + (1/(RRF_K + dense_rank) if present else 0)
    # YOUR CODE: return the top-k items sorted DESC by rrf_score, each with
    # YOUR CODE:   {chunk_id, chunk_text, bm25_rank (or None), dense_rank (or None), rrf_score}
    return []


SAMPLE_QUERY = "How do I update the credit card on my billing account?"
sample_results = hybrid_retrieve(SAMPLE_QUERY, k=50)
print(f"Hybrid retrieve returned {len(sample_results)} results for the sample query.")
if sample_results:
    print(f"Top result: chunk_id={sample_results[0].get('chunk_id')}, "
          f"rrf_score={sample_results[0].get('rrf_score'):.5f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: hybrid_retrieve returns 50 results with correct RRF math.

def checkpoint_3(session):
    try:
        results = hybrid_retrieve(SAMPLE_QUERY, k=50)
        if not isinstance(results, list):
            return CheckpointResult(status="fail",
                error_reason=f"hybrid_retrieve must return a list; got {type(results).__name__}.")
        if len(results) != 50:
            return CheckpointResult(status="fail",
                error_reason=f"hybrid_retrieve returned {len(results)} results; expected exactly 50.")
        required_fields = {"chunk_id", "chunk_text", "bm25_rank", "dense_rank", "rrf_score"}
        for i, r in enumerate(results):
            missing = required_fields - set(r.keys())
            if missing:
                return CheckpointResult(status="fail",
                    error_reason=f"Result {i} missing fields {sorted(missing)}.")
        scores = [r["rrf_score"] for r in results]
        if scores != sorted(scores, reverse=True):
            return CheckpointResult(status="fail",
                error_reason="Results are not sorted DESC by rrf_score.")

        bm25 = bm25_top_n(SAMPLE_QUERY, 50)
        dense = dense_top_n(SAMPLE_QUERY, 50)
        bm25_rank = {x["chunk_id"]: i + 1 for i, x in enumerate(bm25)}
        dense_rank = {x["chunk_id"]: i + 1 for i, x in enumerate(dense)}
        expected = {}
        for cid in set(bm25_rank) | set(dense_rank):
            s = 0.0
            if cid in bm25_rank:
                s += 1.0 / (RRF_K + bm25_rank[cid])
            if cid in dense_rank:
                s += 1.0 / (RRF_K + dense_rank[cid])
            expected[cid] = s

        for r in results[:10]:
            cid = r["chunk_id"]
            exp = expected.get(cid)
            if exp is None:
                return CheckpointResult(status="fail",
                    error_reason=f"Result {cid!r} does not appear in either BM25 or dense top-50; "
                                 f"how did it get fused in?")
            if abs(r["rrf_score"] - exp) > 1e-6:
                return CheckpointResult(status="fail",
                    error_reason=(
                        f"RRF score for {cid!r} is {r['rrf_score']:.6f}; expected {exp:.6f}. "
                        f"Formula is sum(1 / (k + rank)) with k=60, rank 1-indexed; missing-from-list "
                        f"contribution is 0."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=repr(exc))

check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Hybrid retrieval results are wrong. The RRF math is the most common bug. Print the BM25 top-5, dense top-5, and your fused top-5; if a chunk is rank 1 in both BM25 and dense, it must be in your fused top-5 with a higher score than any chunk that appears in only one list.

</details>

<details><summary>Hint 2 (direction)</summary>

The RRF formula is `score(d) = sum_over_lists(1 / (k + rank_in_list(d)))`. `k` is a smoothing parameter, typically 60. `rank_in_list` is 1-indexed; if a document does not appear in a list, its contribution from that list is 0 (NOT `1/k`, NOT skipped from the union).

Build the rank dicts with `enumerate(..., start=1)` or `i + 1` for clarity. Union the chunk IDs across both lists; iterate the union once.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working RRF implementation:

```python
def hybrid_retrieve(query, k=50):
    bm25 = bm25_top_n(query, k)
    dense = dense_top_n(query, k)
    bm25_rank = {x["chunk_id"]: i + 1 for i, x in enumerate(bm25)}
    dense_rank = {x["chunk_id"]: i + 1 for i, x in enumerate(dense)}
    text_lookup = {x["chunk_id"]: x["chunk_text"] for x in bm25}
    for x in dense:
        text_lookup.setdefault(x["chunk_id"], x["chunk_text"])
    fused = []
    for cid in set(bm25_rank) | set(dense_rank):
        score = 0.0
        if cid in bm25_rank:
            score += 1.0 / (RRF_K + bm25_rank[cid])
        if cid in dense_rank:
            score += 1.0 / (RRF_K + dense_rank[cid])
        fused.append({
            "chunk_id": cid,
            "chunk_text": text_lookup[cid],
            "bm25_rank": bm25_rank.get(cid),
            "dense_rank": dense_rank.get(cid),
            "rrf_score": score,
        })
    fused.sort(key=lambda r: r["rrf_score"], reverse=True)
    return fused[:k]
```

</details>

**Wallet check.** One dense query embedding (fractions of a cent) plus one BM25 search (free against the OCU pool). OCU clock is at roughly 40 minutes; OpenSearch at $0.64 cumulative. Running total: about $0.70. The OCU cost dominates everything else in this lab.

## Task 4: Cohere Rerank 3 over the top-50

Reranking is the cheapest quality lever in a retrieval pipeline. You pass the query plus the 50 chunk texts (NOT the dicts) to Cohere, ask for top-5, and map the returned indices back to your chunk IDs.

The checkpoint compares the reranked top-5 against the BM25-only top-5 and the dense-only top-5; if rerank produces the identical set as either baseline, either the wrong inputs were passed or the eval query is too easy. With this synthetic corpus and the sample query, the reranked set should differ from both baselines by at least one chunk.

In [ ]:
# NBVAL_SKIP
# Task 4: rerank the top-50 with Cohere Rerank 3, map indices back to chunk IDs.

co = cohere.ClientV2(api_key=COHERE_API_KEY)


def cohere_rerank_top_n(query, fused_results, top_n=5):
    # YOUR CODE: documents = [r['chunk_text'] for r in fused_results]
    # YOUR CODE: resp = co.rerank(query=query, documents=documents,
    # YOUR CODE:                   top_n=top_n, model=COHERE_RERANK_MODEL)
    # YOUR CODE: for each item in resp.results, map item.index back to
    # YOUR CODE:   fused_results[item.index]['chunk_id']
    # YOUR CODE: return a list of dicts with chunk_id, chunk_text, rerank_score
    return []


reranked_top5 = cohere_rerank_top_n(SAMPLE_QUERY, sample_results, top_n=5)
print("Cohere reranked top-5:")
for r in reranked_top5:
    print(f"  {r.get('chunk_id')}: rerank_score={r.get('rerank_score'):.4f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: rerank returns 5 distinct chunks AND differs from both baselines.

def checkpoint_4(session):
    try:
        bm25 = bm25_top_n(SAMPLE_QUERY, 50)
        dense = dense_top_n(SAMPLE_QUERY, 50)
        fused = hybrid_retrieve(SAMPLE_QUERY, k=50)
        reranked = cohere_rerank_top_n(SAMPLE_QUERY, fused, top_n=5)

        if not isinstance(reranked, list) or len(reranked) != 5:
            n = len(reranked) if isinstance(reranked, list) else type(reranked).__name__
            return CheckpointResult(status="fail",
                error_reason=f"cohere_rerank_top_n must return exactly 5 results; got {n}.")
        rer_ids = [r.get("chunk_id") for r in reranked]
        if len(set(rer_ids)) != 5:
            return CheckpointResult(status="fail",
                error_reason=f"Reranked top-5 has duplicate chunk_ids: {rer_ids}.")

        bm25_top5 = {x["chunk_id"] for x in bm25[:5]}
        dense_top5 = {x["chunk_id"] for x in dense[:5]}
        rer_set = set(rer_ids)
        if rer_set == bm25_top5 or rer_set == dense_top5:
            return CheckpointResult(status="fail",
                error_reason=(
                    f"Reranked top-5 is identical to a baseline top-5; reranker is not doing meaningful work. "
                    f"Confirm you are passing chunk_text strings (not dicts) and top_n=5 to co.rerank()."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=repr(exc))

check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Cohere rerank returned an error or no documents. Check the Cohere dashboard for your credit balance; check the exact model name in the API call.

</details>

<details><summary>Hint 2 (direction)</summary>

The current Cohere rerank model as of 2026-Q2 is `rerank-english-v3.0`. The call shape with the v2 client is:

```
resp = co.rerank(
    query=q,
    documents=[d['chunk_text'] for d in top50],
    top_n=5,
    model='rerank-english-v3.0',
)
```

You pass strings to `documents`, NOT dicts. Each `resp.results` item has an `index` field that maps back to your `top50` list and a `relevance_score` field for the rerank score.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
def cohere_rerank_top_n(query, fused_results, top_n=5):
    documents = [r["chunk_text"] for r in fused_results]
    resp = co.rerank(
        query=query,
        documents=documents,
        top_n=top_n,
        model=COHERE_RERANK_MODEL,
    )
    out = []
    for item in resp.results:
        idx = item.index
        out.append({
            "chunk_id": fused_results[idx]["chunk_id"],
            "chunk_text": fused_results[idx]["chunk_text"],
            "rerank_score": float(item.relevance_score),
        })
    return out
```

</details>

**Wallet check.** One Cohere rerank call billed at $1.00 per 1K = $0.001. Pocket change. OCU is at about 50 minutes cumulative ($0.80). Running total: roughly $0.82. The eval loop in the next task makes 100 more Cohere calls; that adds about 10 cents.

## Task 5: Run the 100-query eval set and clear the 82% recall@5 threshold

You build a deterministic 100-query eval set tied to the synthetic corpus. Each query has one or more "relevant" chunk IDs (the chunks whose topic matches the query's topic). For each query you run the full pipeline (BM25 + dense + RRF top-50 + Cohere rerank top-5) and check whether ANY of the relevant chunks appear in the reranked top-5.

`recall@5 = (queries with at least one relevant chunk in top-5) / (total queries)`

The target is 0.82 on the hybrid+rerank pipeline. The synthetic corpus is generous (30 chunks per topic, queries pulled from the topic set) so the threshold should land at or above 0.85 if the pipeline is correct.

In [ ]:
# NBVAL_SKIP
# Task 5: build the eval set, run the pipeline against each query, compute
# recall@5, write the JSON results file.

random.seed(SEED + 1)
EVAL_QUERY_TEMPLATES = [
    "What is the process for {topic}?",
    "How do I configure {topic} step 5?",
    "Where can I find the {topic} reference note 10?",
    "I am stuck on {topic} settings.",
    "Documentation for {topic} field name.",
    "Common issues with {topic} configuration.",
    "Walk me through {topic} troubleshooting.",
    "Tag lookup for {topic} 03.",
    "{topic} rate limit at 60 requests per minute.",
    "What is the {topic} overview page?",
]

EVAL_QUERIES = []
for topic, _ in TOPICS[:10]:
    for template in EVAL_QUERY_TEMPLATES:
        query_text = template.format(topic=topic)
        relevant_ids = [c["chunk_id"] for c in CORPUS if c["topic"] == topic]
        EVAL_QUERIES.append({"query": query_text, "topic": topic, "relevant_ids": relevant_ids})

print(f"Eval set: {len(EVAL_QUERIES)} queries.")

# YOUR CODE: per_query_rows = []
# YOUR CODE: hits_at_5 = 0
# YOUR CODE: for q in EVAL_QUERIES:
# YOUR CODE:   fused = hybrid_retrieve(q['query'], k=50)
# YOUR CODE:   reranked = cohere_rerank_top_n(q['query'], fused, top_n=5)
# YOUR CODE:   top5_ids = [r['chunk_id'] for r in reranked]
# YOUR CODE:   hit = any(cid in q['relevant_ids'] for cid in top5_ids)
# YOUR CODE:   hits_at_5 += int(hit)
# YOUR CODE:   per_query_rows.append({'query': q['query'], 'top5_ids': top5_ids, 'hit': hit})

# YOUR CODE: recall_at_5 = hits_at_5 / len(EVAL_QUERIES)
# YOUR CODE: results = {
# YOUR CODE:   'summary': {'recall_at_5_hybrid_rerank': recall_at_5,
# YOUR CODE:               'n_queries': len(EVAL_QUERIES),
# YOUR CODE:               'hits_at_5': hits_at_5},
# YOUR CODE:   'per_query': per_query_rows,
# YOUR CODE: }
# YOUR CODE: with open(EVAL_RESULTS_PATH, 'w') as f:
# YOUR CODE:   json.dump(results, f, indent=2)
# YOUR CODE: print(f'recall@5 = {recall_at_5:.3f}')

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: /content/eval_results.json exists; recall_at_5_hybrid_rerank >= 0.82.

def checkpoint_5(session):
    try:
        if not os.path.exists(EVAL_RESULTS_PATH):
            return CheckpointResult(status="fail",
                error_reason=f"{EVAL_RESULTS_PATH} does not exist. The eval loop did not write the JSON.")
        with open(EVAL_RESULTS_PATH) as f:
            data = json.load(f)
        summary = data.get("summary") or {}
        score = summary.get("recall_at_5_hybrid_rerank")
        if score is None:
            return CheckpointResult(status="fail",
                error_reason="summary.recall_at_5_hybrid_rerank missing from results JSON.")
        if score < 0.82:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"recall@5 came in at {score:.3f}; threshold is 0.82. "
                    f"Common causes: (1) chunk IDs in eval labels do not match indexed chunk IDs "
                    f"(confirm chunk_<index> scheme); (2) RRF k mismatched to corpus (try k=10 instead of 60)."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=repr(exc))

check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Recall@5 is below the threshold. Print the per-query hit list and look for patterns; if recall is uniformly low, the issue is systemic, not a few outliers.

</details>

<details><summary>Hint 2 (direction)</summary>

Two suspects: (1) chunk IDs in your eval labels do not match the IDs you assigned at index time. Print one eval label's `relevant_ids` next to one indexed chunk's `chunk_id` and confirm both use the `chunk_<index>` scheme. (2) The RRF k-parameter; try k=10 instead of 60 for more aggressive fusion if the score is close. Use `recall@5 = hits_at_5 / len(EVAL_QUERIES)` exactly; do not normalise by relevant-set size.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5:

```python
per_query_rows = []
hits_at_5 = 0
for q in EVAL_QUERIES:
    fused = hybrid_retrieve(q["query"], k=50)
    reranked = cohere_rerank_top_n(q["query"], fused, top_n=5)
    top5_ids = [r["chunk_id"] for r in reranked]
    hit = any(cid in q["relevant_ids"] for cid in top5_ids)
    hits_at_5 += int(hit)
    per_query_rows.append({
        "query": q["query"],
        "topic": q["topic"],
        "top5_ids": top5_ids,
        "hit": hit,
    })

recall_at_5 = hits_at_5 / len(EVAL_QUERIES)
results = {
    "summary": {
        "recall_at_5_hybrid_rerank": recall_at_5,
        "n_queries": len(EVAL_QUERIES),
        "hits_at_5": hits_at_5,
    },
    "per_query": per_query_rows,
}
with open(EVAL_RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"recall@5 = {recall_at_5:.3f}")
```

</details>

**Wallet check.** Eval loop done. Cohere reranks for 100 queries: ~$0.10. OpenAI embeddings for 100 queries: ~$0.005. OpenSearch is at roughly 75 minutes cumulative ($1.20) and still ticking until cleanup. Running total: about $1.32. Cleanup time.

## Cleanup

Critical-first teardown: delete the OpenSearch Serverless collection (this stops the OCU clock immediately), wait for `DELETED`, then drop the three security policies, the EventBridge rule, the Lambda function, the IAM role, the Supabase pgvector table, and the local eval results JSON.

The verification scan re-queries each resource. The tag audit scans for any AWS resource carrying `labrun:lab-slug=hybrid-search-bm25-dense-rerank` and any Supabase table whose name starts with the lab prefix. A dirty cleanup triggers `sys.exit(1)` so the companion panel surfaces the incomplete state.

In [ ]:
# NBVAL_SKIP
# Cleanup. Critical-first per RESOURCE-SAFETY-SPEC.md Section 2 Rule 2.
# Collection deletion polls up to 5 minutes for the DELETED state. The
# canonical summary follows Rule 6.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(1 for r in CLEANUP_MANIFEST if r.critical and r.id in failed_ids)
standard_destroyed = standard_total - sum(1 for r in CLEANUP_MANIFEST if not r.critical and r.id in failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $1.50 to $2.00.** OpenSearch Serverless ~$1.30 (90 minutes of OCU). OpenAI embeddings ~$0.05 (300 corpus + 100 eval queries). Cohere rerank ~$0.10 (100 eval calls plus a few sanity reranks). Watchdog Lambda + EventBridge: free at this scale. Supabase pgvector: free tier. Cleanup deleted the collection so OCU billing stopped at the moment the cleanup cell finished. Check your AWS billing console in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. The RRF k-parameter changes the balance between BM25 and dense contributions. You ran with k=60. If the lead asked you to switch to k=10, what do you predict happens to recall@5 on the same eval set, and why? Walk through what a lower k does to a chunk that ranks 1 in BM25 but does not appear in dense at all.

2. The team's existing system uses dense-only at 62% recall@5. Your hybrid+rerank pipeline scored above 82%. The lead asks: "Is the 20-point lift worth the added complexity of running OpenSearch Serverless alongside pgvector?" Write the two sentences you would send back. Include the dollar number for OpenSearch Serverless's monthly minimum at 4 OCUs.

## Exam-Style Practice

**Question 1 (retrieval stack latency):**

A team's RAG pipeline retrieves 50 chunks via hybrid search, reranks the top-50 with Cohere Rerank 3, and returns top-5 to the LLM. The p95 retrieval latency is 380ms; the team needs to land it under 200ms. Which change is most likely to cut latency the most without dropping recall meaningfully?

A. Drop the reranker and use RRF-only top-5.
B. Reduce the rerank candidate set from top-50 to top-25.
C. Swap Cohere Rerank for a self-hosted reranker on Modal.
D. Pre-compute the reranking offline.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: removes the largest single quality lever; latency drops but recall@5 falls by ~15 points on most corpora. The whole point of the rerank stage is the quality lift.
- B is correct: rerank latency scales roughly linearly with candidate count; cutting candidates in half halves the rerank stage, which is the biggest single contributor to p95. Recall@5 typically degrades less than 1 point at top-25 candidates.
- C is wrong direction here: self-hosting adds infra complexity and cold-start risk; latency usually does not improve over the managed API; only justified if data residency is the driver.
- D is wrong: reranking is query-dependent and cannot be pre-computed without knowing the query ahead of time.

</details>

**Question 2 (RRF math):**

A document `chunk_42` appears at rank 1 in the BM25 list and at rank 8 in the dense list. Using reciprocal rank fusion with k=60 and 1-indexed ranks, what is the document's RRF score?

A. 1/61 + 1/68
B. 1/1 + 1/8
C. 1/60 + 1/8
D. 1/61

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: RRF score = sum over lists of `1 / (k + rank_in_list)`. With k=60 and ranks 1 and 8 respectively, that is `1/(60+1) + 1/(60+8) = 1/61 + 1/68`.
- B is wrong: omits the smoothing constant; this is the "1/rank" formula, which is rank-reciprocal but NOT reciprocal RANK fusion.
- C is wrong: applies k to the first term but not the second; both ranks get the same k.
- D is wrong: ignores the dense-list contribution entirely; both lists contribute additively.

</details>

**Question 3 (when reranking is wasted spend):**

For which retrieval scenario is the cost of a learned reranker hardest to justify?

A. A 100K-document corpus where users ask varied, ambiguous questions.
B. A 50-document corpus of FAQ entries with a small, well-known query set.
C. A code-search system where queries are syntactically diverse.
D. A multilingual support corpus where users query in any of 12 languages.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong (rerank is justified): large corpus + ambiguous queries is exactly where rerank lifts recall meaningfully; the rerank API cost amortises across many queries with high quality lift.
- B is correct (rerank is wasted): a 50-doc FAQ with a known query distribution can be served at top-5 by either BM25 or dense alone; rerank cost recurs per query for negligible quality gain. The team should benchmark each baseline first.
- C is wrong (rerank is justified): code search benefits heavily from reranking because BM25 alone tokenises code badly and dense embeddings miss exact syntactic matches.
- D is wrong (rerank is justified): multilingual retrieval benefits from reranking because cross-lingual dense embeddings have known recall gaps that a learned reranker can patch.

</details>